In [ ]:
import pandas as pd

# Load the dataset from a CSV file
def load_dataset(file_path):
    dataset = pd.read_csv(file_path)

    # For this example, assuming 'article' is the column containing the articles
    # Adjust this based on the actual column names in your dataset
    dataset = dataset.rename(columns={"article": "text"})

    return dataset

# Specify the file path for your dataset
file_path = "/content/train.csv"

# Load the dataset
dataset = load_dataset(file_path)

# Display the first few rows of the dataset
print(dataset.head())

In [ ]:
import pandas as pd

def load_dataset(file_path):
    try:
        # Try loading the dataset
        dataset = pd.read_csv(file_path)
        return dataset
    except pd.errors.ParserError:
        # If there's a parsing error, handle it
        print("Parsing error encountered. Attempting to clean the data...")
        with open(file_path, 'r') as file:
            lines = file.readlines()

        # Clean the data (example: remove quotes)
        cleaned_lines = [line.replace('"', '') for line in lines]

        # Write cleaned data to a temporary file
        temp_file_path = 'temp.csv'
        with open(temp_file_path, 'w') as temp_file:
            temp_file.writelines(cleaned_lines)

        # Load the cleaned dataset
        dataset = pd.read_csv(temp_file_path)

        # Remove temporary file
        import os
        os.remove(temp_file_path)

        return dataset

# Example usage:
file_path = '/content/test.csv'
dataset = load_dataset(file_path)
print(dataset.head())


In [ ]:
import csv

In [ ]:
import nltk
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
# Define the preprocess_text function
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    sentences = sent_tokenize(text)
    preprocessed_sentences = []
    for sent in sentences:
        words = word_tokenize(sent)
        words = [word.lower() for word in words if word.isalnum() and word.lower() not in stop_words]
        preprocessed_sentences.append(" ".join(words))
    return preprocessed_sentences

In [ ]:
# Apply TextRank algorithm for text summarization
def textrank_summary(text, num_sentences=3):
    preprocessed_sentences = preprocess_text(text)

    sentence_similarity = np.zeros((len(preprocessed_sentences), len(preprocessed_sentences)))
    for i in range(len(preprocessed_sentences)):
        for j in range(len(preprocessed_sentences)):
            if i != j:
                sentence1 = set(preprocessed_sentences[i].split())
                sentence2 = set(preprocessed_sentences[j].split())
                similarity_score = len(sentence1.intersection(sentence2)) / (np.log(len(sentence1)) + np.log(len(sentence2)) + 1e-6)
                sentence_similarity[i][j] = similarity_score

    scores = np.ones(len(preprocessed_sentences))
    damping_factor = 0.85
    for _ in range(100):
        new_scores = (1 - damping_factor) + damping_factor * np.sum(sentence_similarity, axis=1)
        if np.allclose(new_scores, scores, rtol=1e-4):
            break
        scores = new_scores

    top_sentences_idx = np.argsort(-scores)[:num_sentences]
    summary = [preprocessed_sentences[i] for i in top_sentences_idx]

    return " ".join(summary)


In [ ]:
# Apply TextRank summarization to the dataset
train['text_column'] = train['article'].apply(lambda x: textrank_summary(x, num_sentences=3))
train.to_csv('dataset_with_summaries_text_rank.csv', index=False)
print("Summaries saved to 'dataset_with_summaries_text_rank.csv'.")


In [ ]:
# Ensure 'article' column is string-like by converting to string
dataset['article'] = dataset['article'].astype(str)


In [ ]:
# Define the preprocess_text function
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    sentences = sent_tokenize(text)
    preprocessed_sentences = []
    for sent in sentences:
        words = word_tokenize(sent)
        words = [word.lower() for word in words if word.isalnum() and word.lower() not in stop_words]
        preprocessed_sentences.append(" ".join(words))
    return preprocessed_sentences

# Apply TextRank algorithm for text summarization
def textrank_summary(text, num_sentences=3):
    preprocessed_sentences = preprocess_text(text)

    sentence_similarity = np.zeros((len(preprocessed_sentences), len(preprocessed_sentences)))
    for i in range(len(preprocessed_sentences)):
        for j in range(len(preprocessed_sentences)):
            if i != j:
                sentence1 = set(preprocessed_sentences[i].split())
                sentence2 = set(preprocessed_sentences[j].split())
                similarity_score = len(sentence1.intersection(sentence2)) / (np.log(len(sentence1)) + np.log(len(sentence2)) + 1e-6)
                sentence_similarity[i][j] = similarity_score

    scores = np.ones(len(preprocessed_sentences))
    damping_factor = 0.85
    for _ in range(100):
        new_scores = (1 - damping_factor) + damping_factor * np.sum(sentence_similarity, axis=1)
        if np.allclose(new_scores, scores, rtol=1e-4):
            break
        scores = new_scores

    top_sentences_idx = np.argsort(-scores)[:num_sentences]
    summary = [preprocessed_sentences[i] for i in top_sentences_idx]

    return " ".join(summary)

In [ ]:
# Apply TextRank summarization to the dataset
dataset['text_column'] = dataset['article'].apply(lambda x: textrank_summary(x, num_sentences=3))
train.to_csv('dataset_with_summaries_text_rank.csv', index=False)
print("Summaries saved to 'dataset_with_summaries_text_rank.csv'.")


In [ ]:
# Apply TextRank summarization to the dataset
dataset['text_column'] = dataset['article'].apply(lambda x: textrank_summary(x, num_sentences=3))
dataset.to_csv('dataset_with_summaries_text_rank.csv', index=False)
print("Summaries saved to 'dataset_with_summaries_text_rank.csv'.")

In [ ]:
# Display the first few rows of the dataset with summaries
print(train.head())

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

nltk.download('stopwords')
nltk.download('punkt')

import pandas as pd

# Function to load the dataset with error handling
def load_dataset(file_path):
    try:
        dataset = pd.read_csv(file_path, error_bad_lines=False)  # Skip problematic rows
    except pd.errors.ParserError as e:
        print(f"Error while parsing CSV: {e}")
        # Handle the error as needed, e.g., skip the row or display an error message
        dataset = pd.DataFrame()  # Or provide an empty DataFrame

    return dataset

# Specify the file path for your dataset
file_path = "test.csv"

# Load the dataset with error handling
dataset = load_dataset(file_path)

if not dataset.empty:
    # Data processing steps go here
    print(dataset.head())

# Data Preprocessing
stop_words = set(stopwords.words('english'))

# Function to clean text
def clean_text(text):
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub('"', '', text)
    text = ' '.join([word for word in text.split() if word.lower() not in stop_words])
    return text

df_cnn['cleaned_article'] = df_cnn['article'].apply(clean_text)

# Tokenization
def tokenize_sentences(article):
    sentences = sent_tokenize(article)
    return sentences

# Function to generate article summary
def generate_summary(article, n=5):
    sentences = tokenize_sentences(article)
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(sentences)
    similarity_matrix = cosine_similarity(X, X)
    nx_graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(nx_graph)
    ranked_sentences = sorted(((scores[i], sentence) for i, sentence in enumerate(sentences)), reverse=True)
    summary_sentences = [sentence for _, sentence in ranked_sentences[:n]]
    return ' '.join(summary_sentences)

# Function to get the article and its summary based on ID
def get_summary_by_id(dataframe, id):
    row = dataframe[dataframe['id'] == id]
    if not row.empty:
        article = row['article'].values[0]
        summary = generate_summary(article)
        return article, summary
    else:
        return "ID not found in the dataset", ""

# Example usage
article_id = '0054d6d30dbcad772e20b22771153a2a9cbeaf62'
article, summary = get_summary_by_id(df_cnn, article_id)
print("Article:\n", article)
print("\nSummary:\n", summary)

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

nltk.download('stopwords')
nltk.download('punkt')

# Function to load the dataset with error handling
def load_dataset(file_path):
    try:
        dataset = pd.read_csv(file_path, error_bad_lines=False)  # Skip problematic rows
    except pd.errors.ParserError as e:
        print(f"Error while parsing CSV: {e}")
        # Handle the error as needed, e.g., skip the row or display an error message
        dataset = pd.DataFrame()  # Or provide an empty DataFrame

    return dataset

# Specify the file path for your dataset
file_path = "test.csv"

# Load the dataset with error handling
dataset = load_dataset(file_path)

if not dataset.empty:
    # Data preprocessing: Cleaning the article text
    stop_words = set(stopwords.words('english'))

    # Function to clean text
    def clean_text(text):
        text = re.sub(r'\([^)]*\)', '', text)
        text = re.sub('"', '', text)
        text = ' '.join([word for word in text.split() if word.lower() not in stop_words])
        return text

    dataset['cleaned_article'] = dataset['article'].apply(clean_text)

    # Tokenization
    def tokenize_sentences(article):
        sentences = sent_tokenize(article)
        return sentences

    # Function to generate article summary
    def generate_summary(article, n=5):
        sentences = tokenize_sentences(article)
        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(sentences)
        similarity_matrix = cosine_similarity(X, X)
        nx_graph = nx.from_numpy_array(similarity_matrix)
        scores = nx.pagerank(nx_graph)
        ranked_sentences = sorted(((scores[i], sentence) for i, sentence in enumerate(sentences)), reverse=True)
        summary_sentences = [sentence for _, sentence in ranked_sentences[:n]]
        return ' '.join(summary_sentences)

    # Function to get the article and its summary based on ID
    def get_summary_by_id(dataframe, id):
        row = dataframe[dataframe['id'] == id]
        if not row.empty:
            article = row['article'].values[0]
            summary = generate_summary(article)
            return article, summary
        else:
            return "ID not found in the dataset", ""

    # Example usage
    article_id = '0054d6d30dbcad772e20b22771153a2a9cbeaf62'
    article, summary = get_summary_by_id(dataset, article_id)
    print("Article:\n", article)
    print("\nSummary:\n", summary)

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

nltk.download('stopwords')
nltk.download('punkt')

# Function to load the dataset with error handling
def load_dataset(file_path):
    try:
        dataset = pd.read_csv(file_path, error_bad_lines=False, skiprows=range(10863, None))  # Skip problematic rows
    except pd.errors.ParserError as e:
        print(f"Error while parsing CSV: {e}")
        # Handle the error as needed, e.g., skip the row or display an error message
        dataset = pd.DataFrame()  # Or provide an empty DataFrame

    return dataset

# Specify the file path for your dataset
file_path = "test.csv"

# Load the dataset with error handling
dataset = load_dataset(file_path)

if not dataset.empty:
    # Data preprocessing: Cleaning the article text
    stop_words = set(stopwords.words('english'))

    # Function to clean text
    def clean_text(text):
        text = re.sub(r'\([^)]*\)', '', text)
        text = re.sub('"', '', text)
        text = ' '.join([word for word in text.split() if word.lower() not in stop_words])
        return text

    dataset['cleaned_article'] = dataset['article'].apply(clean_text)

    # Tokenization
    def tokenize_sentences(article):
        sentences = sent_tokenize(article)
        return sentences

    # Function to generate article summary
    def generate_summary(article, n=5):
        sentences = tokenize_sentences(article)
        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(sentences)
        similarity_matrix = cosine_similarity(X, X)
        nx_graph = nx.from_numpy_array(similarity_matrix)
        scores = nx.pagerank(nx_graph)
        ranked_sentences = sorted(((scores[i], sentence) for i, sentence in enumerate(sentences)), reverse=True)
        summary_sentences = [sentence for _, sentence in ranked_sentences[:n]]
        return ' '.join(summary_sentences)

    # Function to get the article and its summary based on ID
    def get_summary_by_id(dataframe, id):
        row = dataframe[dataframe['id'] == id]
        if not row.empty:
            article = row['article'].values[0]
            summary = generate_summary(article)
            return article, summary
        else:
            return "ID not found in the dataset", ""

    # Example usage
    article_id = '0054d6d30dbcad772e20b22771153a2a9cbeaf62'
    article, summary = get_summary_by_id(dataset, article_id)
    print("Article:\n", article)
    print("\nSummary:\n", summary)

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

nltk.download('stopwords')
nltk.download('punkt')

# Function to load the dataset with error handling
def load_dataset(file_path):
    try:
        dataset = pd.read_csv(file_path, error_bad_lines=False, skiprows=lambda x: x >= 10863)  # Skip rows starting from 10863
    except pd.errors.ParserError as e:
        print(f"Error while parsing CSV: {e}")
        # Handle the error as needed, e.g., skip the row or display an error message
        dataset = pd.DataFrame()  # Or provide an empty DataFrame

    return dataset

# Specify the file path for your dataset
file_path = "test.csv"

# Load the dataset with error handling
dataset = load_dataset(file_path)

if not dataset.empty:
    # Data preprocessing: Cleaning the article text
    stop_words = set(stopwords.words('english'))

    # Function to clean text
    def clean_text(text):
        text = re.sub(r'\([^)]*\)', '', text)
        text = re.sub('"', '', text)
        text = ' '.join([word for word in text.split() if word.lower() not in stop_words])
        return text

    dataset['cleaned_article'] = dataset['article'].apply(clean_text)

    # Tokenization
    def tokenize_sentences(article):
        sentences = sent_tokenize(article)
        return sentences

    # Function to generate article summary
    def generate_summary(article, n=5):
        sentences = tokenize_sentences(article)
        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(sentences)
        similarity_matrix = cosine_similarity(X, X)
        nx_graph = nx.from_numpy_array(similarity_matrix)
        scores = nx.pagerank(nx_graph)
        ranked_sentences = sorted(((scores[i], sentence) for i, sentence in enumerate(sentences)), reverse=True)
        summary_sentences = [sentence for _, sentence in ranked_sentences[:n]]
        return ' '.join(summary_sentences)

    # Function to get the article and its summary based on ID
    def get_summary_by_id(dataframe, id):
        row = dataframe[dataframe['id'] == id]
        if not row.empty:
            article = row['article'].values[0]
            summary = generate_summary(article)
            return article, summary
        else:
            return "ID not found in the dataset", ""

    # Example usage
if not dataset.empty:
    article_id = '92c514c913c0bdfe25341af9fd72b29db544099b'
    article, summary = get_summary_by_id(dataset, article_id)

    # Counting words in the article and summary
    article_word_count = count_words(article)
    summary_word_count = count_words(summary)

    print("Article Word Count:", article_word_count)
    print("Summary Word Count:", summary_word_count)


In [ ]:
from nltk.tokenize import word_tokenize
import pandas as pd

In [ ]:
import pandas as pd
import networkx as nx
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
import re
from string import punctuation
from heapq import nlargest

# Load the dataset
df = pd.read_csv("test.csv")

# Contractions dictionary and regex
contractions_dict = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he had",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how'll": "how will",
    "how's": "how is",
    "i'd": "i would",
    "i'd've": "i would have",
    "i'll": "i will",
    "i'll've": "i will have",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it'd": "it would",
    "it'd've": "it would have",
    "it'll": "it will",
    "it'll've": "it will have",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "o'clock": "of the clock",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shan't've": "shall not have",
    "she'd": "she would",
    "she'd've": "she would have",
    "she'll": "she will",
    "she'll've": "she will have",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so is",
    "that'd": "that would",
    "that'd've": "that would have",
    "that's": "that is",
    "there'd": "there would",
    "there'd've": "there would have",
    "there's": "there is",
    "they'd": "they would",
    "they'd've": "they would have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they're": "they are",
    "they've": "they have",
    "to've": "to have",
    "wasn't": "was not",
    "we'd": "we would",
    "we'd've": "we would have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what'll've": "what will have",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "when's": "when is",
    "when've": "when have",
    "where'd": "where did",
    "where's": "where is",
    "where've": "where have",
    "who'll": "who will",
    "who'll've": "who will have",
    "who's": "who is",
    "who've": "who have",
    "why's": "why is",
    "why've": "why have",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y?all": "you all",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "you'd": "you would",
    "you'd've": "you would have",
    "you'll": "you will",
    "you'll've": "you will have",
    "you're": "you are",
    "you're": "you are",
    "you've": "you have",
}

contractions_re = re.compile('(%s)' % '|'.join(contractions_dict.keys()))

# Function to clean the HTML from the article

In [ ]:
# Function to clean HTML tags from the article
def clean_html(raw_html):
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, '', raw_html)
    return cleantext

# Function to expand contractions
def expand_contractions(s, contractions_dict=contractions_dict):
    def replace(match):
        return contractions_dict[match.group(0)]
    return contractions_re.sub(replace, s)

In [ ]:
# Preprocessing function
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = clean_html(text)  # Remove HTML tags
    text = expand_contractions(text)  # Expand contractions
    text = re.sub(r'\S+@\S+', '', text)  # Remove email addresses
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)  # Remove URLs
    text = text.replace("\xa0", " ")  # Remove '\xa0'
    text = text.replace("'s", '')  # Strip possessives
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespaces
    return text


In [ ]:
# Function to preprocess the entire article
def preprocess_article(article):
    return preprocess_text(article)

# Function to preprocess the article and tokenize it into sentences
def preprocess_and_tokenize(article):
    sentences = sent_tokenize(preprocess_article(article))
    return sentences

In [ ]:
# Function to create a graph from sentences
def create_graph(sentences):
    graph = nx.Graph()
    graph.add_nodes_from(range(len(sentences)))
    vectorizer = CountVectorizer().fit_transform(sentences)
    similarity_matrix = cosine_similarity(vectorizer)
    for i in range(len(sentences)):
        for j in range(i+1, len(sentences)):
            graph.add_edge(i, j, weight=similarity_matrix[i][j])
    return graph

In [ ]:
# Function to rank sentences using TextRank algorithm
def text_rank(graph):
    scores = nx.pagerank(graph)
    return scores

# Function to generate summary
def generate_summary(article, scores, top_n=3):
    ranked_sentences = sorted(((scores[i], s) for i, s in enumerate(article)), reverse=True)
    summary = ' '.join([sentence[1] for sentence in ranked_sentences[:top_n]])
    return summary

# Main function
def main(article_id):
    # Get article based on article_id
    article = df.loc[df['id'] == article_id]['article'].values[0]

    # Preprocess the article and tokenize into sentences
    preprocessed_sentences = preprocess_and_tokenize(article)

    # Create graph from sentences
    graph = create_graph(preprocessed_sentences)

    # Rank sentences using TextRank algorithm
    scores = text_rank(graph)

    # Generate summary
    summary = generate_summary(preprocessed_sentences, scores)

    # Return article and summary
    return article, summary

# Test the function
article_id = '92c514c913c0bdfe25341af9fd72b29db544099b'  # Example ID
article, summary = main(article_id)

# Count the number of words in article and summary
article_word_count = len(article.split())
summary_word_count = len(summary.split())

print("Article:\n", article)
print("\nSummary:\n", summary)
print("\nNumber of words in article:", article_word_count)
print("Number of words in summary:", summary_word_count)

In [ ]:
pip install rouge


In [ ]:
from rouge import Rouge

# Function to calculate ROUGE scores
def calculate_rouge(reference_summary, generated_summary):
    rouge = Rouge()
    scores = rouge.get_scores(generated_summary, reference_summary)
    return scores

# Reference summary (highlight) from the dataset
reference_summary = df.loc[df['id'] == article_id]['highlights'].values[0]

# Calculate ROUGE scores
rouge_scores = calculate_rouge(reference_summary, summary)

# Print ROUGE scores
print("ROUGE scores:")
print(rouge_scores)


In [ ]:
import textwrap
from transformers import pipeline

In [ ]:
# Test the function
article_id = '92c514c913c0bdfe25341af9fd72b29db544099b'  # Example ID
article, summary = main(article_id)

# Count the number of words in article and summary
article_word_count = len(article.split())
summary_word_count = len(summary.split())

print("Article:\n", article)
print("\nSummary:\n", summary)
print("\nNumber of words in article:", article_word_count)
print("Number of words in summary:", summary_word_count)

# Word wrap article and summary
wrapper = textwrap.TextWrapper(width=80)
article_wrapped = wrapper.fill(text=article)
summary_wrapped = wrapper.fill(text=summary)

# Print article, summary, and word counts
print("Article:\n", article_wrapped)
print("\nSummary:\n", summary_wrapped)

from rouge import Rouge

# Function to calculate ROUGE scores
def calculate_rouge(reference_summary, generated_summary):
    rouge = Rouge()
    scores = rouge.get_scores(generated_summary, reference_summary)
    return scores

# Reference summary (highlight) from the dataset
reference_summary = df.loc[df['id'] == article_id]['highlights'].values[0]

# Calculate ROUGE scores
rouge_scores = calculate_rouge(reference_summary, summary)

# Print ROUGE scores
print("ROUGE scores:")
print(rouge_scores)


In [ ]:
from transformers import pipeline

# Load the BERT-based text summarization pipeline
summarization_pipeline = pipeline("summarization")

# Function to generate summary using BERT
def generate_summary_bert(article):
    summarized_text = summarization_pipeline(article, max_length=150, min_length=30, do_sample=False)
    summary = summarized_text[0]['summary_text']
    return summary

# Main function with BERT summarization
def main_bert(article_id):
    # Get article based on article_id
    article = df.loc[df['id'] == article_id]['article'].values[0]

    # Generate summary using BERT
    summary = generate_summary_bert(article)

    # Return article and summary
    return article, summary

# Test the BERT-based summarization function
article_id = '92c514c913c0bdfe25341af9fd72b29db544099b'  # Example ID
article, summary_bert = main_bert(article_id)

# Print the BERT-generated summary
print("\nBERT-generated Summary:\n", summary_bert)


In [ ]:
from transformers import pipeline

# Load the BERT-based text summarization pipeline
summarization_pipeline = pipeline("summarization")

# Function to generate summary using BERT
def generate_summary_bert(article):
    summarized_text = summarization_pipeline(article, max_length=150, min_length=30, do_sample=False)
    summary = summarized_text[0]['summary_text']
    return summary

# Main function with BERT summarization
def main_bert(article_id):
    # Get article based on article_id
    article = df.loc[df['id'] == article_id]['article'].values[0]

    # Generate summary using BERT
    summary = generate_summary_bert(article)

    # Count the number of words in article and summary
    article_word_count = len(article.split())
    summary_word_count = len(summary.split())

    # Print article, summary, and word counts
    print("Article:\n", article)
    print("\nSummary:\n", summary)
    print("\nNumber of words in article:", article_word_count)
    print("Number of words in summary:", summary_word_count)

# Test the BERT-based summarization function
article_id = '92c514c913c0bdfe25341af9fd72b29db544099b'  # Example ID
main_bert(article_id)


In [ ]:
import textwrap
from transformers import pipeline

# Load the BERT-based text summarization pipeline
summarization_pipeline = pipeline("summarization")

# Function to generate summary using BERT
def generate_summary_bert(article):
    summarized_text = summarization_pipeline(article, max_length=150, min_length=30, do_sample=False)
    summary = summarized_text[0]['summary_text']
    return summary

# Main function with BERT summarization
def main_bert(article_id):
    # Get article based on article_id
    article = df.loc[df['id'] == article_id]['article'].values[0]

    # Generate summary using BERT
    summary = generate_summary_bert(article)

    # Count the number of words in article and summary
    article_word_count = len(article.split())
    summary_word_count = len(summary.split())

    # Word wrap article and summary
    wrapper = textwrap.TextWrapper(width=80)
    article_wrapped = wrapper.fill(text=article)
    summary_wrapped = wrapper.fill(text=summary)

    # Print article, summary, and word counts
    print("Article:\n", article_wrapped)
    print("\nSummary:\n", summary_wrapped)
    print("\nNumber of words in article:", article_word_count)
    print("Number of words in summary:", summary_word_count)

# Test the BERT-based summarization function
article_id = '92c514c913c0bdfe25341af9fd72b29db544099b'  # Example ID
main_bert(article_id)


In [ ]:
import textwrap
from transformers import pipeline
from rouge import Rouge

# Load the BERT-based text summarization pipeline
summarization_pipeline = pipeline("summarization")

# Function to generate summary using BERT
def generate_summary_bert(article):
    summarized_text = summarization_pipeline(article, max_length=150, min_length=30, do_sample=False)
    summary = summarized_text[0]['summary_text']
    return summary

# Main function with BERT summarization
def main_bert(article_id):
    # Get article based on article_id
    article = df.loc[df['id'] == article_id]['article'].values[0]

    # Generate summary using BERT
    summary = generate_summary_bert(article)

    # Count the number of words in article and summary
    article_word_count = len(article.split())
    summary_word_count = len(summary.split())

    # Word wrap article and summary
    wrapper = textwrap.TextWrapper(width=80)
    article_wrapped = wrapper.fill(text=article)
    summary_wrapped = wrapper.fill(text=summary)

    # Print article, summary, and word counts
    print("Article:\n", article_wrapped)
    print("\nSummary:\n", summary_wrapped)
    print("\nNumber of words in article:", article_word_count)
    print("Number of words in summary:", summary_word_count)

    # Calculate ROUGE scores
    rouge = Rouge()
    rouge_scores = rouge.get_scores(summary, df.loc[df['id'] == article_id]['highlights'].values[0])
    print("\nROUGE scores:")
    print(rouge_scores)

# Test the BERT-based summarization function
article_id = '92c514c913c0bdfe25341af9fd72b29db544099b'  # Example ID
main_bert(article_id)


In [ ]:
import textwrap
from transformers import pipeline
from rouge import Rouge

# Load the BERT-based text summarization pipeline
summarization_pipeline = pipeline("summarization")

# Function to generate summary using BERT
def generate_summary_bert(article):
    # Adjust max_length and min_length parameters to control summary length
    summarized_text = summarization_pipeline(article, max_length=300, min_length=150, do_sample=False)
    summary = summarized_text[0]['summary_text']
    return summary

# Main function with BERT summarization
def main_bert(article_id):
    # Get article based on article_id
    article = df.loc[df['id'] == article_id]['article'].values[0]

    # Generate summary using BERT
    summary = generate_summary_bert(article)

    # Count the number of words in article and summary
    article_word_count = len(article.split())
    summary_word_count = len(summary.split())

    # Word wrap article and summary
    wrapper = textwrap.TextWrapper(width=80)
    article_wrapped = wrapper.fill(text=article)
    summary_wrapped = wrapper.fill(text=summary)

    # Print article, summary, and word counts
    print("Article:\n", article_wrapped)
    print("\nSummary:\n", summary_wrapped)
    print("\nNumber of words in article:", article_word_count)
    print("Number of words in summary:", summary_word_count)

    # Calculate ROUGE scores
    rouge = Rouge()
    rouge_scores = rouge.get_scores(summary, df.loc[df['id'] == article_id]['highlights'].values[0])
    print("\nROUGE scores:")
    print(rouge_scores)

# Test the BERT-based summarization function
article_id = '92c514c913c0bdfe25341af9fd72b29db544099b'  # Example ID
main_bert(article_id)


In [ ]:
import pandas as pd
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the dataset
file_path = "test.csv"
dataset = pd.read_csv(file_path)

# Load GPT-2 model and tokenizer
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

# Function for text summarization using GPT-2 model
def generate_summary_gpt2(text):
    inputs = gpt2_tokenizer.encode("summarize: " + text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = gpt2_model.generate(inputs, max_length=450, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = gpt2_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Find article and summary based on ID '92c514c913c0bdfe25341af9fd72b29db544099b'
article_gpt2 = dataset.loc[dataset['id'] == '92c514c913c0bdfe25341af9fd72b29db544099b']['article'].values[0]
summary_gpt2 = generate_summary_gpt2(article_gpt2)

# Print the article and summary
print("GPT-2 Summary:")
print("\nArticle:\n", article_gpt2)
print("\nSummary:\n", summary_gpt2)

In [ ]:
import pandas as pd

# Load the dataset
file_path = "test.csv"
dataset = pd.read_csv(file_path)

# Display column names
print("Column names in the dataset:")
print(dataset.columns)


In [ ]:
import pandas as pd
import textwrap
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the dataset
file_path = "test.csv"
dataset = pd.read_csv(file_path)

# Load GPT-2 model and tokenizer
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

# Function for text summarization using GPT-2 model
def generate_summary_gpt2(text):
    inputs = gpt2_tokenizer.encode("summarize: " + text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = gpt2_model.generate(inputs, max_length=450, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = gpt2_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Function to wrap text and count words
def wrap_and_count(text):
    wrapped_text = textwrap.fill(text, width=80)
    word_count = len(text.split())
    return wrapped_text, word_count

# Find article and summary based on ID '92c514c913c0bdfe25341af9fd72b29db544099b'
article_id = '92c514c913c0bdfe25341af9fd72b29db544099b'
article_gpt2 = dataset.loc[dataset['id'] == article_id]['article'].values[0]
summary_gpt2 = generate_summary_gpt2(article_gpt2)

# Wrap article and summary and count words
wrapped_article, article_word_count = wrap_and_count(article_gpt2)
wrapped_summary, summary_word_count = wrap_and_count(summary_gpt2)

# Print the article, summary, and word count
print("GPT-2 Summary:")
print("\nArticle:\n", wrapped_article)
print("\nSummary:\n", wrapped_summary)
print("\nNumber of words in article:", article_word_count)
print("Number of words in summary:", summary_word_count)


In [ ]:
import pandas as pd
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from rouge import Rouge

# Load the dataset
file_path = "test.csv"
dataset = pd.read_csv(file_path)

# Load GPT-2 model and tokenizer
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

# Function for text summarization using GPT-2 model
def generate_summary_gpt2(text):
    inputs = gpt2_tokenizer.encode("summarize: " + text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = gpt2_model.generate(inputs, max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = gpt2_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Main function to process each article in the dataset
def process_articles(dataset):
    for index, row in dataset.iterrows():
        article = row['article']
        summary = generate_summary_gpt2(article)
        print("GPT-2 Summary:\n", summary)

# Call the main function
process_articles(dataset)


In [ ]:
import pandas as pd
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from rouge import Rouge

# Load the dataset
file_path = "test.csv"
dataset = pd.read_csv(file_path)

# Load GPT-2 model and tokenizer
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

# Function for text summarization using GPT-2 model
def generate_summary_gpt2(text):
    inputs = gpt2_tokenizer.encode("summarize: " + text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = gpt2_model.generate(inputs, max_length=1000, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = gpt2_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Main function to process each article in the dataset
def process_articles(dataset):
    for index, row in dataset.iterrows():
        article = row['article']
        summary = generate_summary_gpt2(article)
        print("GPT-2 Summary:\n", summary)

# Call the main function
process_articles(dataset)
